# GENCO-Δ on GOC 2000 (Small): conditioning a grid foundation model on the pre-contingency state

**Question.** At contingency time the operator already holds the solved pre-contingency (base-case) state. The released
GENCO power-flow task zeroes every masked input. **GENCO-Δ** fills the masked inputs with that base-case information
(bus Vm/Va, optionally the pre-contingency branch flows) and fine-tunes the model to predict the post-contingency state.
Does that make GENCO a better Newton warm start than the base case itself? On the 2,000-bus grid the base case currently wins
(about 2.95 vs 3.23 Newton iterations), so this is the hard test.

**What this notebook does** (GPU runtime, about 1.5 to 3 hours with the defaults, mostly fine-tuning and CPU Newton solves):

1. installs the Python dependencies (pinned `gridfm-graphkit` commit) and unpacks our code (you upload a tarball, our repo is private);
2. downloads GOC 2000 partitions 0 to 19 (4,000 scenarios, about 180 load scenarios) from the Hugging Face Hub;
3. builds a **split by load scenario** (validation and test load scenarios are ones the released checkpoint never saw) and the **base-case store**;
4. fine-tunes the released Small seed-0 checkpoint in three arms, with identical data, epochs and seed:
   `control` (original PowerFlow task), `delta_vm_va` (masked Vm/Va filled with the base case), `delta_vm_va_flows` (plus pre-contingency flows);
5. evaluates the released checkpoint and each arm: Newton iterations to 1e-2 / 1e-4 / 1e-6 MW from the model output, from the base case and from a flat start; state error; element misses after 1 and 2 Newton steps; cluster-bootstrap 95% CIs over load scenarios;
6. zips everything for download.

Run the cells in order. Set `DRY_RUN = True` first for a 10 to 15 minute end-to-end check.

In [ ]:
# ---------------- parameters: edit here, then run this cell (re-run it after a session restart) ----------------
DRY_RUN = False                      # True: 1 epoch on 3% of the batches, 2 evaluation batches, tag prefix "dry_" (pipeline check)
ARMS = ["control", "delta_vm_va", "delta_vm_va_flows"]
EPOCHS = 6                           # fine-tuning epochs per arm (each epoch is about 2 to 4 minutes on a T4)
BATCH_SIZE = 8                       # the released config uses 8; lower it if the GPU runs out of memory
LR = 1e-4
MAX_TRAIN_MINUTES = 100              # wall-clock cap per arm; the best checkpoint so far is kept
N_PARTITIONS = 20                    # 200 scenarios each -> 4000 scenarios
N_VAL_LOAD, N_TEST_LOAD = 12, 24     # unseen load scenarios (about 22 scenarios each) for validation / test; at most about 40 exist in 20 partitions
SPLIT_SEED = 0
MAX_EVAL_BATCHES = None              # None = whole test split; an int limits it (each batch has BATCH_SIZE scenarios)
FORCE = False                        # True: redo stages whose outputs already exist
SAVE_TO_DRIVE = False                # True: mount Google Drive and copy results there after each stage (survives a disconnect)
INCLUDE_WEIGHTS = True               # put the fine-tuned weights (about 25 MB each) into the final zip

TARBALL = "/content/genco_repo.tgz"  # created on your machine, see the next step
REPO = "/content/genco"
GRID, SIZE = "case2000_goc", "small"

In [ ]:
import subprocess, sys
print(sys.version)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv", shell=True, capture_output=True, text=True).stdout
      or "NO GPU FOUND: Runtime > Change runtime type > T4 GPU (or better)")

## 1. Upload our code

The GitHub repository is private, so Colab cannot clone it. On your own machine, in the repository root, run

```bash
./scripts/pack_for_colab.sh
```

It writes `dist/genco_colab_bundle.tar.gz` (about 0.2 MB: `src/`, `scripts/`, `configs/`, a few tests, the tiny split files and a `BUNDLE_INFO.txt`; no data, checkpoints or results). This notebook uses `src/` (`genco_certified`) and `scripts/` (training, evaluation and store-building scripts).
Then run the next cell and pick the tarball in the file dialog; whatever it is called, it is stored at the path in `TARBALL` and unpacked into `REPO`. Alternatively copy the file to Google Drive, mount Drive
and set `TARBALL = "/content/drive/MyDrive/genco_colab_bundle.tar.gz"` in the parameters cell. The same bundle serves `colab_gpu_timing.ipynb`; see `notebooks/README.md`.

In [ ]:
import os, shutil, tarfile
from pathlib import Path

if not (Path(REPO) / "src/genco_certified/__init__.py").exists():
    if not Path(TARBALL).exists():
        try:
            from google.colab import files          # opens a file picker
        except ImportError:
            raise FileNotFoundError(f"{TARBALL} not found and this is not Colab: unpack the bundle into {REPO} yourself")
        up = files.upload()                         # pick dist/genco_colab_bundle.tar.gz (built by scripts/pack_for_colab.sh)
        assert up, "no file chosen"
        shutil.move(next(iter(up)), TARBALL)        # whatever the file was called
    Path(REPO).mkdir(parents=True, exist_ok=True)
    with tarfile.open(TARBALL) as tf:
        try:
            tf.extractall(REPO, filter="data")
        except TypeError:                           # Python < 3.12
            tf.extractall(REPO)
    hits = list(Path(REPO).rglob("src/genco_certified/__init__.py"))
    assert hits, "the tarball must contain src/genco_certified (build it with scripts/pack_for_colab.sh)"
    top = hits[0].parents[2]                        # tolerate a wrapper folder inside the tarball
    if top != Path(REPO):
        for item in top.iterdir():
            shutil.move(str(item), REPO)

need = ["src/genco_certified/graphkit_plugin.py", "src/genco_certified/basecase_store.py", "src/genco_certified/acpf.py",
        "src/genco_certified/loading.py", "scripts/finetune.py", "scripts/eval_genco_delta.py", "scripts/build_basecase_store.py"]
missing = [f for f in need if not (Path(REPO) / f).exists()]
assert not missing, f"missing in the tarball: {missing}"
print("repository unpacked at", REPO)
info = Path(REPO) / "BUNDLE_INFO.txt"
print(info.read_text() if info.exists() else "(no BUNDLE_INFO.txt: tarball not built by scripts/pack_for_colab.sh?)")

## 2. Install dependencies

`gridfm-graphkit` is pinned to the commit our results were produced with. It needs `torch` between 2.10 and 2.14; Colab may ship an older one,
in which case pip upgrades it. `torch-scatter` has no wheel for the newest torch versions: if none is available we install a small
pure-PyTorch stand-in (`scatter_add`, `scatter_mean`, `scatter_max`, which is all the code uses). That is slower than the compiled op but only
by a few percent of the step time.

**If the cell reports `RESTART REQUIRED`:** use *Runtime > Restart session*, then re-run the parameters cell and continue from section 3
(you can skip the upload and install cells; they are re-entrant).

In [ ]:
import importlib.metadata as md, subprocess, sys
from pathlib import Path

GRAPHKIT = "gridfm-graphkit @ git+https://github.com/gridfm/gridfm-graphkit@8a8defbc9ab41caa65b07c75e8e1ee23636877cd"
SHIM = Path("/content/shims")

def sh(cmd, check=True):
    print("$", cmd, flush=True)
    rc = subprocess.run(cmd, shell=True).returncode
    if check and rc:
        raise RuntimeError(f"command failed ({rc}): {cmd}")
    return rc

def ver(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None

def torch_ok():
    v = ver("torch")
    if not v:
        return False
    major, minor = (int(x) for x in v.split("+")[0].split(".")[:2])
    return (2, 10) <= (major, minor) < (2, 14)

before = {p: ver(p) for p in ("torch", "numpy", "pandas", "gridfm-graphkit")}

# 1. torch in the range gridfm-graphkit supports
if not torch_ok():
    sh('pip install -q "torch>=2.10,<2.14" torchvision torchaudio')

# 2. torch-scatter: wheel from data.pyg.org if one exists for this torch/CUDA, else a pure-torch stand-in
def can_import(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

tv, cu = subprocess.run([sys.executable, "-c", "import torch;print(torch.__version__.split('+')[0], torch.version.cuda or 'cpu')"],
                        capture_output=True, text=True).stdout.split()
have_scatter = can_import("torch_scatter") and not (SHIM / "torch_scatter").exists()
if not have_scatter:
    tag = "cu" + cu.replace(".", "") if cu != "cpu" else "cpu"
    sh(f"pip install -q --only-binary=:all: torch-scatter -f https://data.pyg.org/whl/torch-{tv}+{tag}.html", check=False)
    have_scatter = can_import("torch_scatter")
if not have_scatter:
    SHIM_SRC = """
# Pure-PyTorch stand-in for the three torch_scatter functions gridfm-graphkit uses.
import torch

def _size(src, index, dim, dim_size):
    if dim_size is None:
        dim_size = int(index.max()) + 1 if index.numel() else 0
    size = list(src.shape)
    size[dim] = dim_size
    return size

def _expand(src, index, dim):
    if index.dim() == 1 and src.dim() > 1:
        shape = [1] * src.dim()
        shape[dim] = -1
        index = index.view(shape).expand_as(src)
    return index

def scatter_add(src, index, dim=-1, out=None, dim_size=None):
    dim = dim % src.dim()
    if out is None:
        out = torch.zeros(_size(src, index, dim, dim_size), dtype=src.dtype, device=src.device)
    if index.dim() == 1:
        return out.index_add_(dim, index, src)
    return out.scatter_add_(dim, _expand(src, index, dim), src)

def scatter_mean(src, index, dim=-1, out=None, dim_size=None):
    total = scatter_add(src, index, dim, None, dim_size)
    count = scatter_add(torch.ones_like(src), index, dim, None, dim_size)
    return total / count.clamp(min=1)

def scatter_max(src, index, dim=-1, out=None, dim_size=None):
    dim = dim % src.dim()
    low = torch.finfo(src.dtype).min
    init = torch.full(_size(src, index, dim, dim_size), low, dtype=src.dtype, device=src.device)
    vals = init.scatter_reduce(dim, _expand(src, index, dim), src, reduce="amax", include_self=True)
    vals = torch.where(vals == low, torch.zeros_like(vals), vals)
    return vals, torch.zeros_like(vals, dtype=torch.long)
"""
    (SHIM / "torch_scatter").mkdir(parents=True, exist_ok=True)
    (SHIM / "torch_scatter/__init__.py").write_text(SHIM_SRC)
    print("torch_scatter: using the pure-PyTorch stand-in in", SHIM)

# 3. gridfm-graphkit (no dependency resolution against torch) and its remaining dependencies
if ver("gridfm-graphkit") is None:
    sh(f'pip install -q --no-deps --ignore-requires-python "{GRAPHKIT}"')
sh('pip install -q --ignore-requires-python "pandas>=2.3,<3" "numpy>=2.2.6" scipy pyarrow huggingface_hub '
   '"mlflow>=3.1.0" nbformat networkx opt-einsum plotly pyyaml "torch-geometric>=2.6.1" lightning seaborn '
   '"urllib3>=2.6.0" "gdown>=6.0.0" "gridfm-datakit==1.1.0"')

after = {p: ver(p) for p in before}
changed = {p: (before[p], after[p]) for p in ("torch", "numpy", "pandas") if before[p] != after[p] and before[p] is not None}
print("\ninstalled:", after)
if changed:
    print("\n" + "=" * 78)
    print("RESTART REQUIRED: pip changed", changed)
    print("Runtime > Restart session, then re-run the parameters cell and continue with section 3.")
    print("=" * 78)
else:
    print("no restart needed")

## 3. Environment check

Run this after the (optional) restart. It points the scripts at the unpacked repository and checks the GPU.

In [ ]:
import os, sys, time, json, shutil, subprocess
from pathlib import Path

SHIM = "/content/shims"
paths = [f"{REPO}/src"] + ([SHIM] if Path(SHIM, "torch_scatter").exists() else [])
os.environ["PYTHONPATH"] = os.pathsep.join(paths + [os.environ.get("PYTHONPATH", "")]).rstrip(os.pathsep)
sys.path[:0] = paths
os.environ.update(GENCO_CERTIFIED_ROOT=REPO, MLFLOW_ALLOW_FILE_STORE="true", HF_HUB_DISABLE_PROGRESS_BARS="1")
os.chdir(REPO)

import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type"
import genco_certified.loading, genco_certified.graphkit_plugin            # noqa: F401  (registers the new task)
from gridfm_graphkit.io.registries import TASK_REGISTRY
assert "PowerFlowContingencyAware" in TASK_REGISTRY

PREFIX = "dry_" if DRY_RUN else ""
LIMIT_TRAIN = 0.03 if DRY_RUN else 1.0
EPOCHS_RUN = 1 if DRY_RUN else EPOCHS
EVAL_BATCHES = 2 if DRY_RUN else MAX_EVAL_BATCHES
DATA = Path(REPO) / "data/pf" / GRID
OUT = Path(REPO) / "results/genco_delta"
(OUT / "logs").mkdir(parents=True, exist_ok=True)

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
def checkpoint_to_drive(note=""):
    if SAVE_TO_DRIVE:
        dst = Path("/content/drive/MyDrive/genco_delta_goc2000")
        shutil.copytree(OUT, dst / "results", dirs_exist_ok=True)
        if (DATA / "splits_delta").exists():
            shutil.copytree(DATA / "splits_delta", dst / "splits_delta", dirs_exist_ok=True)
        print("copied results to", dst, note)

def run(args, log, keep=("fit done", "saved", "run record", "Error", "error", "Traceback", "->", "scenarios", "built", "wrote", "WARNING", "iters", "state error")):
    """Run a command in the repo root, write all output to `log`, echo the informative lines, raise on failure."""
    log = Path(log); log.parent.mkdir(parents=True, exist_ok=True)
    t0, last = time.time(), time.time()
    print("$", " ".join(map(str, args)), flush=True)
    with open(log, "w") as f:
        p = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=REPO)
        for line in p.stdout:
            f.write(line); f.flush()
            if any(k in line for k in keep):
                print("   ", line.rstrip()[:200], flush=True)
            elif time.time() - last > 120:
                print(f"    ... running, {time.time() - t0:.0f}s", flush=True); last = time.time()
        rc = p.wait()
    print(f"    exit {rc} after {time.time() - t0:.0f}s (log: {log})", flush=True)
    if rc:
        print("".join(open(log).readlines()[-40:]))
        raise RuntimeError(f"failed: {args[:3]}")

## 4. Data: GOC 2000 partitions 0 to 19

Graphkit's streaming loader needs scenario ids `0..N-1`, so the contiguous prefix of partitions is fetched (200 scenarios each).

In [ ]:
import pandas as pd
from huggingface_hub import snapshot_download

TABLES = ["bus_data", "gen_data", "branch_data"]
if FORCE or not (DATA / "raw/bus_data.parquet").exists() or len(list((DATA / "raw/bus_data.parquet").glob("scenario_partition=*"))) < N_PARTITIONS:
    patterns = [f"{t}.parquet/scenario_partition={i}/*" for t in TABLES for i in range(N_PARTITIONS)]
    snapshot_download(f"gridfm/pf_small_{GRID}", repo_type="dataset", local_dir=DATA / "raw", allow_patterns=patterns + ["n_scenarios.txt", "args.log"])
ids = pd.read_parquet(DATA / "raw/bus_data.parquet", columns=["scenario"]).scenario.unique()
assert ids.min() == 0 and ids.max() == len(ids) - 1, (ids.min(), ids.max(), len(ids))
print(len(ids), "scenarios on disk")

# the released checkpoint's own published split (which scenarios it trained on)
snapshot_download("gridfm/genco-pf-datakit", local_dir=Path(REPO) / "checkpoints/genco-pf-datakit",
                  allow_patterns=[f"mlflow/*/{GRID}/{SIZE}/seed0/*/artifacts/stats/{GRID}_scenario_splits.json"])
print(list((Path(REPO) / "checkpoints/genco-pf-datakit").glob(f"mlflow/*/{GRID}/{SIZE}/seed0/*/artifacts/stats/*.json")))

## 5. Split by load scenario, with untouched validation and test sets

All topology variants of one load scenario share the same base case, so they stay together. The released checkpoint was pre-trained on a random 20,000-scenario
subset of the whole dataset (about 8% of the variants of a typical load scenario), so only about 40 of the 182 load scenarios in these partitions are **completely unseen** by it
(no variant in its published train or validation split). Those unseen load scenarios are used for **test and validation**; training uses all other load scenarios
(partial overlap with pre-training data, identical for all arms). The last load scenario is dropped because the partition cut truncates it.

In [ ]:
import glob, numpy as np, torch

def make_delta_splits(grid_dir, out_dir, n_val, n_test, seed, grid=GRID, size=SIZE, repo=REPO):
    pub = json.load(open(sorted(glob.glob(f"{repo}/checkpoints/genco-pf-datakit/mlflow/*/{grid}/{size}/seed0/*/artifacts/stats/{grid}_scenario_splits.json"))[0]))
    b = pd.read_parquet(Path(grid_dir) / "raw/bus_data.parquet", columns=["scenario", "load_scenario_idx"]).drop_duplicates("scenario")
    b["load_scenario_idx"] = b.load_scenario_idx.astype(int)
    ls_of = b.set_index("scenario").load_scenario_idx
    seen = set(ls_of.reindex([i for i in pub["train"] + pub["val"] if i in ls_of.index]).dropna().astype(int))
    last = int(b.load_scenario_idx.max())
    unseen = sorted(set(b.load_scenario_idx) - seen - {last})
    assert len(unseen) >= n_val + n_test, f"only {len(unseen)} load scenarios are unseen by the released model; lower N_VAL_LOAD / N_TEST_LOAD"
    perm = np.random.default_rng(seed).permutation(unseen)
    parts = {"test": list(perm[:n_test]), "val": list(perm[n_test:n_test + n_val]),
             "train": list(perm[n_test + n_val:]) + sorted(seen - {last})}
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    info = {"seed": seed, "n_load_scenarios_total": int(b.load_scenario_idx.nunique()), "n_seen_by_released_model": len(seen),
            "n_unseen": len(unseen), "dropped_last": last}
    for k, ls in parts.items():
        sids = sorted(b[b.load_scenario_idx.isin(set(map(int, ls)))].scenario.tolist())
        torch.save(torch.tensor(sids, dtype=torch.long), out_dir / f"{k}.pt")
        info[k] = {"n_load_scenarios": int(len(ls)), "n_scenarios": len(sids)}
    # invariants: disjoint by load scenario; validation and test never seen by the released model
    s = {k: set(map(int, v)) for k, v in parts.items()}
    assert not (s["train"] & s["val"] or s["train"] & s["test"] or s["val"] & s["test"])
    assert not ((s["val"] | s["test"]) & seen)
    (out_dir / "split_info.json").write_text(json.dumps(info, indent=1))
    return info

SPLIT = DATA / "splits_delta"
if FORCE or not (SPLIT / "test.pt").exists():
    info = make_delta_splits(DATA, SPLIT, N_VAL_LOAD, N_TEST_LOAD, SPLIT_SEED)
else:
    info = json.load(open(SPLIT / "split_info.json"))
print(json.dumps(info, indent=1))

## 6. Base-case store

For every load scenario the intact-topology AC power flow is solved from all of its topology variants (`acpf.base_case_voltages`; if the
flat start diverges, which happens for about 6% of GOC 2000 load scenarios, Newton is restarted from a variant's stored solution).
The store maps every scenario id to `(Vm [p.u.], Va [rad])`. It takes a few minutes.

In [ ]:
store_path = DATA / "basecase_store.pt"
if FORCE or not store_path.exists():
    run([sys.executable, "scripts/build_basecase_store.py", "--data_root", "data/pf", "--grid", GRID, "--force"], OUT / "logs/build_store.log",
        keep=("built", "wrote", "WARNING", "probe", "Error"))
from genco_certified.basecase_store import BaseCaseStore
store = BaseCaseStore.load(store_path)
print(store.meta)
for k in ("train", "val", "test"):
    assert store.covers(torch.load(SPLIT / f"{k}.pt").tolist()), k
checkpoint_to_drive("(store + splits)")

## 7. Fine-tune the arms

Same data (the train split above), same released initialisation, same epochs, learning rate, batch size and training seed; the checkpoint with the lowest
**validation** loss is kept (the test split is never used for selection). Each arm writes `results/genco_delta/runs/<tag>.json`
(data/split/store hashes, versions, validation curve, best epoch). The released normaliser is reused (graphkit's own fine-tune command would refit it).
With `DRY_RUN = True` this takes about a minute per arm.

In [ ]:
ARM_FLAGS = {
    "control": ["--task_name", "PowerFlow"],
    "delta_vm_va": ["--task_name", "PowerFlowContingencyAware", "--delta_inputs", "vm_va"],
    "delta_vm_va_flows": ["--task_name", "PowerFlowContingencyAware", "--delta_inputs", "vm_va_flows"],
}
max_time = f"00:{MAX_TRAIN_MINUTES // 60:02d}:{MAX_TRAIN_MINUTES % 60:02d}:00"
for arm in ARMS:
    tag = f"{PREFIX}{arm}_s0"
    if (OUT / "runs" / f"{tag}.json").exists() and not FORCE:
        print("skip", tag, "(run record exists)"); continue
    run([sys.executable, "scripts/finetune.py", *ARM_FLAGS[arm], "--grid", GRID, "--size", SIZE, "--seed", 0, "--seed_train", 0,
         "--data_root", "data/pf", "--split", "splits_delta", "--epochs", EPOCHS_RUN, "--lr", LR, "--batch_size", BATCH_SIZE,
         "--accelerator", "gpu", "--limit_train_batches", LIMIT_TRAIN, "--max_time", max_time, "--tag", tag,
         "--log_dir", OUT / "train_logs" / tag],
        OUT / f"logs/train_{tag}.log")
    rec = json.load(open(OUT / "runs" / f"{tag}.json"))
    print(f"   {tag}: best epoch {rec['selection']['best_epoch']} of {rec['training']['epochs_run']}, val loss {rec['selection']['best_val_loss']:.3g}, "
          f"{rec['training']['wall_seconds']:.0f}s")
    checkpoint_to_drive(tag)

## 8. Evaluate

The released checkpoint (task `PowerFlow`, released weights) and each arm are scored on the held-out test split by the same script. For each scenario:
Newton iterations to 1e-2 / 1e-4 / 1e-6 MW from the model output (voltage setpoints enforced), from the base case and from a flat start;
state error against the datakit solution; element decisions (overloads above 1.01 of the rating, PQ-bus voltage outside the limits by more than 1 mV)
after 0, 1 and 2 Newton steps from the model and 1 and 2 steps from the base case. The Newton solves run on the CPU, so this is the slow stage (about 10 minutes per model on the default split).

In [ ]:
EV = OUT / "eval"
jobs = [("released", ["--task_name", "PowerFlow"], None)]
for arm in ARMS:
    jobs.append((arm, ARM_FLAGS[arm], f"checkpoints/finetuned/{PREFIX}{arm}_s0/best_model_state_dict.pt"))
for name, flags, weights in jobs:
    tag = f"{PREFIX}{name}_s0__goc2000"
    if (EV / f"{tag}.json").exists() and not FORCE:
        print("skip", tag); continue
    cmd = [sys.executable, "scripts/eval_genco_delta.py", *flags, "--grid", GRID, "--size", SIZE, "--seed", 0, "--data_root", "data/pf",
           "--split", "splits_delta", "--batch_size", BATCH_SIZE, "--tag", tag]
    if weights:
        cmd += ["--weights", weights]
    if EVAL_BATCHES:
        cmd += ["--max_batches", EVAL_BATCHES]
    run(cmd, OUT / f"logs/eval_{tag}.log")
    checkpoint_to_drive(tag)

## 9. Results

Mean Newton iterations (95% cluster-bootstrap CI over load scenarios) for the starts "base case" and "flat start" (identical for every model, they do not depend on it) and for the model outputs. Lower is better.
The row to look at is **model − base**: a negative value with a confidence interval below zero means the model is a better warm start than the pre-contingency state.

In [ ]:
sys.path.insert(0, f"{REPO}/scripts")
import eval_genco_delta as E
import matplotlib.pyplot as plt

S = {}
for f in sorted(EV.glob(f"{PREFIX}*_s0__goc2000.json")):
    S[f.stem.split("__")[0].removeprefix(PREFIX).removesuffix("_s0")] = json.load(open(f))
ci = lambda d: f"{d['mean']:.2f} [{d['ci95'][0]:.2f}, {d['ci95'][1]:.2f}]"
rows = []
ref = next(iter(S.values()))
for e in (2, 4, 6):
    rows.append({"start": "base case", "tol": f"1e-{e}", "iterations": ci(ref[f"nr_iters_b_e{e}"])})
    rows.append({"start": "flat start", "tol": f"1e-{e}", "iterations": ci(ref[f"nr_iters_f_e{e}"])})
    for name, s in S.items():
        rows.append({"start": name, "tol": f"1e-{e}", "iterations": ci(s[f"nr_iters_g_e{e}"]),
                     "model - base": ci(s[f"nr_iters_g_minus_b_e{e}"]),
                     "share with fewer iterations than base": f"{s[f'frac_fewer_iters_than_base_e{e}']['mean']:.0%}"})
tbl = pd.DataFrame(rows).fillna("")
print(f"{ref['n_scenarios']} test scenarios, {ref['n_load_scenarios']} load scenarios")
from IPython.display import display
display(tbl)

print("\nmedian state error (l2 over the unknowns) and element misses after k Newton steps (truth: %d violating elements)" % ref["truth_elements"])
rows2 = []
for name, s in S.items():
    rows2.append({"model": name, "state err (model)": f"{s['err_l2_g']['median']['median']:.3g}", "state err (base case)": f"{s['err_l2_b']['median']['median']:.3g}",
                  **{f"misses {m}": s[f"{m}_el_miss_count"] for m in E.DECISIONS}, **{f"false alarms {m}": s[f"{m}_el_fa_count"] for m in ("g1", "b1")}})
print(pd.DataFrame(rows2).to_string(index=False))

fig, ax = plt.subplots(figsize=(7.5, 3.6))
names = ["flat start", "base case"] + list(S)
for j, e in enumerate((2, 4, 6)):
    vals = [ref[f"nr_iters_f_e{e}"], ref[f"nr_iters_b_e{e}"]] + [s[f"nr_iters_g_e{e}"] for s in S.values()]
    x = np.arange(len(names)) + (j - 1) * 0.27
    ax.bar(x, [v["mean"] for v in vals], width=0.26, label=f"to 1e-{e} MW",
           yerr=[[v["mean"] - v["ci95"][0] for v in vals], [v["ci95"][1] - v["mean"] for v in vals]], capsize=2)
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=20, ha="right"); ax.set_ylabel("Newton iterations"); ax.legend(frameon=False)
ax.set_title("GOC 2000 (Small): Newton iterations from each start")
plt.tight_layout(); fig.savefig(OUT / f"{PREFIX}goc2000_iterations.png", dpi=150); plt.show()
E.aggregate(EV, PREFIX)

## 10. Download

Everything needed to analyse the runs offline: per-scenario parquet and summary JSON for each model, run records, logs, split ids, the base-case store and (optionally) the fine-tuned weights.

In [ ]:
import zipfile
zip_path = Path(REPO).parent / f"{PREFIX}genco_delta_goc2000_results.zip"
extra = [DATA / "splits_delta", DATA / "basecase_store.pt", Path(REPO) / "results/configs"]
if INCLUDE_WEIGHTS:
    extra += [Path(REPO) / "checkpoints/finetuned" / f"{PREFIX}{a}_s0" / "best_model_state_dict.pt" for a in ARMS]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root in [OUT] + [p for p in extra if p.exists()]:
        for f in ([root] if root.is_file() else sorted(root.rglob("*"))):
            if f.is_file():
                z.write(f, f.relative_to(REPO))
print(zip_path, f"{zip_path.stat().st_size / 1e6:.0f} MB")
checkpoint_to_drive("(final)")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:                       # large browser downloads can fail: use Drive or the file browser
    print("download via the Files panel (left) or set SAVE_TO_DRIVE = True:", e)

## Troubleshooting

- **CUDA out of memory** while fine-tuning: lower `BATCH_SIZE` (4) and re-run section 7 with `FORCE = True`.
- **`torch_scatter` messages**: section 2 falls back to a pure-PyTorch stand-in automatically; nothing to do.
- **Session disconnected**: re-run the parameters cell, section 1 (the tarball may still be in `/content`), 3, 4, 5, 6; finished stages are skipped
  (their outputs exist). With `SAVE_TO_DRIVE = True` copy `genco_delta_goc2000/results` from Drive back to `/content/genco/results/genco_delta` first.
- **Training too slow**: reduce `EPOCHS`, or `MAX_TRAIN_MINUTES` (training stops at the cap and keeps the best checkpoint so far).
- **`PowerFlowContingencyAware` not registered / import errors**: the tarball must contain `src/genco_certified/` from the same revision as `scripts/`.
- **Reading the numbers**: all comparisons use the same test scenarios and the same Newton routine. The base-case row is the industry-standard warm start,
  the released row is GENCO as published. A delta arm is only interesting if it beats the base case *and* the `control` arm (same fine-tuning, no base-case input).